# jmedqa Analysis Notebook

このノートブックは `jmedqa_pred_light.csv` を読み込み、
- 全体/年度別/セクション別/臨床領域別正解率
- 抽出LLMの改善・劣化率
- 抽出違反率
を確認します。

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# 解析対象ディレクトリを変更
INPUT_DIR = Path('outputs_jmedqa_extract_llm')  # 例: outputs_jmedqa_re
PATTERN = 'jmedqa_pred_light.csv'

csv_files = sorted(INPUT_DIR.glob(f'**/{PATTERN}'))
print('files:', len(csv_files))
for p in csv_files[:10]:
    print('-', p)

In [ ]:
dfs = []
for p in csv_files:
    d = pd.read_csv(p)
    d['run'] = str(p.parent.relative_to(INPUT_DIR))
    if 'is_correct' in d.columns:
        d['is_correct'] = pd.to_numeric(d['is_correct'], errors='coerce').fillna(0).astype(int)
    dfs.append(d)

df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print('rows:', len(df), 'cols:', len(df.columns))
df.head(3)

In [ ]:
if len(df) == 0:
    raise ValueError('No rows loaded.')

overall = (
    df.groupby('run', as_index=False)['is_correct']
      .mean()
      .rename(columns={'is_correct': 'acc'})
)
display(overall.sort_values('acc', ascending=False))

if {'prediction_stage1_direct', 'prediction', 'gold_answer'}.issubset(df.columns):
    tmp = df.copy()
    tmp['stage1_is_correct'] = (tmp['prediction_stage1_direct'].fillna('').astype(str) == tmp['gold_answer'].fillna('').astype(str)).astype(int)
    tmp['extractor_is_correct'] = (tmp['prediction'].fillna('').astype(str) == tmp['gold_answer'].fillna('').astype(str)).astype(int)
    tmp['improved'] = ((tmp['stage1_is_correct'] == 0) & (tmp['extractor_is_correct'] == 1)).astype(int)
    tmp['degraded'] = ((tmp['stage1_is_correct'] == 1) & (tmp['extractor_is_correct'] == 0)).astype(int)
    effect = tmp.groupby('run', as_index=False).agg(
        stage1_acc=('stage1_is_correct', 'mean'),
        extractor_acc=('extractor_is_correct', 'mean'),
        improved_rate=('improved', 'mean'),
        degraded_rate=('degraded', 'mean'),
    )
    effect['delta_acc'] = effect['extractor_acc'] - effect['stage1_acc']
    display(effect.sort_values('delta_acc', ascending=False))

In [ ]:
def plot_acc_by(df, key, topn=None, figsize=(10,4)):
    if key not in df.columns:
        print(f'skip: {key} not found')
        return
    g = df.groupby(['run', key], as_index=False)['is_correct'].mean().rename(columns={'is_correct': 'acc'})
    if topn is not None:
        keep = (g.groupby(key)['acc'].mean().sort_values(ascending=False).head(topn).index)
        g = g[g[key].isin(keep)]
    plt.figure(figsize=figsize)
    sns.barplot(data=g, x=key, y='acc', hue='run')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.title(f'Accuracy by {key}')
    plt.tight_layout()
    plt.show()

plot_acc_by(df, 'year')
plot_acc_by(df, 'section')
plot_acc_by(df, 'clinical_area', topn=15, figsize=(12,5))
plot_acc_by(df, 'answer_mode')
plot_acc_by(df, 'is_calc')

In [ ]:
violation_cols = [c for c in df.columns if c.startswith('violation_')]
if violation_cols:
    v = df.groupby('run', as_index=False)[violation_cols].mean()
    display(v)

    vv = v.melt(id_vars=['run'], value_vars=violation_cols, var_name='metric', value_name='rate')
    plt.figure(figsize=(12,5))
    sns.barplot(data=vv, x='metric', y='rate', hue='run')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.title('Violation Rates by Run')
    plt.tight_layout()
    plt.show()
else:
    print('No violation columns found.')